In [149]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(os.getcwd()))))
sys.path.append(os.path.abspath(os.path.join("../..", "src")))

In [150]:
import numpy as np
import torch

from data.data_generator import ExampleScalarFieldTwoDimensions, ExampleScalarFieldThreeDimensions
from data.data_generator import ExampleVectorialFieldThreeDimensions, ExampleVectorialFieldTwoDimensions

In [151]:
import tensopsciml as to

from tensopsciml import TensOps
from tutorials.utils.utilities import mae

### Parameters

In [152]:
N_data = 100
dx = dy = dz = 0.1

indexing='ij'

In [153]:
alpha = np.random.rand()
beta = np.random.rand()
gamma = np.random.rand()

### First order tensors (vectors)

In [154]:
u_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='sin(x) + cos(y)', Fy_char='(z) - sin(x)', Fz_char='cos(x + y + z)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func().to(dtype=torch.float32)

v_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='sin(x**2) + cos(y**3)', Fy_char='(z) + sin(x)', Fz_char='cos(x*y*z)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func().to(dtype=torch.float32)

w_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='sin(x) * cos(y) + tan(z)', Fy_char='sin(x**2 + y**2)', Fz_char='cos(z) - sin(x*y)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func().to(dtype=torch.float32)

x_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='sin(x + y) * cos(z)', Fy_char='sin(x) - cos(y)', Fz_char='tan(z) + sin(x + y)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func().to(dtype=torch.float32)

y_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='cos(z)', Fy_char='sin(x)', Fz_char='sin(x + y)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func().to(dtype=torch.float32)

z_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='cos(x)', Fy_char='sin(z)', Fz_char='sin(z*y)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func().to(dtype=torch.float32)


o_ = torch.zeros_like(u_)
O_ = o_[:, 0].unsqueeze(1)
i_ = torch.ones_like(O_)

In [155]:
u = TensOps(u_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)
v = TensOps(v_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)
w = TensOps(w_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)
x = TensOps(x_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)
y = TensOps(y_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)
z = TensOps(z_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)

o = TensOps(o_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)
O = TensOps(O_, domain_dim=3, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None)
i = TensOps(i_, domain_dim=3, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None)

### Metric

We work in $\mathbb{R}^3$ with the standard Euclidean metric $g_{ij} = \delta_{ij}$. The library never fabricates a metric internally, so we build it explicitly here and pass it to every operation that changes tensor variance (`lower_index`, `raise_index`, `kronecker_delta`, `cross_prod`, `levi_civita_tensor`, ...).

In [156]:
discretization = list(u_.shape[-3:])

metric_ = torch.eye(3, dtype=u_.dtype).reshape(1, 3, 3, 1, 1, 1).expand(N_data, 3, 3, *discretization).clone()
metric = to.MetrOps(metric_, domain_dim=3, codomain_dim=[3, 3])
inv_metric = to.inverse(metric)

### Levi-Civita tensor

The raw Levi-Civita symbol is a tensor density, not a tensor: scaling it by the metric's volume form turns it into the genuine covariant Levi-Civita tensor $E_{ijk}$ that `cross_prod` needs for 3-component vectors.

In [157]:
e = to.levi_civita_symbol(N_data, 3, discretization)
E = to.levi_civita_tensor(e, domain_dim=3, codomain_dim=[3, 3, 3], order=[0, 3], metric=metric)

### Second order tensors (matrices)

In [158]:
A_ = torch.stack((u_, v_, w_), dim=2)
B_ = torch.stack((w_, v_, x_), dim=2)
C_ = torch.stack((w_, x_, x_), dim=2)
D_ = torch.stack((v_, v_, x_), dim=2)

OO_ = torch.stack((o_, o_, o_), dim=2)

In [159]:
A = TensOps(A_, domain_dim=3, codomain_dim=[3, 3], order=[1, 1], contravariant_indices=[0], covariant_indices=[1])
B = TensOps(B_, domain_dim=3, codomain_dim=[3, 3], order=[1, 1], contravariant_indices=[0], covariant_indices=[1])
C = TensOps(C_, domain_dim=3, codomain_dim=[3, 3], order=[1, 1], contravariant_indices=[0], covariant_indices=[1])
D = TensOps(D_, domain_dim=3, codomain_dim=[3, 3], order=[1, 1], contravariant_indices=[0], covariant_indices=[1])

OO = TensOps(OO_, domain_dim=3, codomain_dim=[3, 3], order=[1, 1], contravariant_indices=[0], covariant_indices=[1])
I = to.kronecker_delta(metric)

## 1.1 Algebra of vectors

This identities are obtained from *Holzapfel, G. A. (2002). Nonlinear solid mechanics: a continuum approach for engineering science.* They are the different identities of first chapter

$(1.3)$ $$u^i + v^i = v^i + u^i$$

In [160]:
left = u + v
right = v + u

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.4)$ $$(u^i + v^i) + w^i = u^i + (v^i + w^i)$$

In [161]:
left = (u + v) + w
right = u + (v + w)

print(f"MAE = {mae(left, right)}")

MAE = 6.488087933576026e-08


$(1.5)$ $$u^i + o^i = u^i$$

In [162]:
left = u + o
right = u

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.6)$ $$ u^i + (-u^i) = o^i$$

In [163]:
left = u + (- u)
right = o

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.7)$ $$ (\alpha \cdot \beta)\cdot u^i = \alpha \cdot (\beta \cdot u^i)  $$

In [164]:
left = (alpha*beta)*u
right = alpha*(beta*u)

print(f"MAE = {mae(left, right)}")

MAE = 1.048439446371674e-09


$(1.8)$ $$ (\alpha + \beta) \cdot u^i = \alpha \cdot u^i + \beta \cdot u^i  $$

In [165]:
left = (alpha + beta)*u
right = alpha*u + beta*u

print(f"MAE = {mae(left, right)}")

MAE = 3.639765111529414e-08


$(1.9)$  $$ \alpha \cdot (u^i + v^i) = \alpha \cdot u^i + \alpha \cdot v^i  $$

In [166]:
left = alpha*(u + v)
right = alpha*u + alpha*v

print(f"MAE = {mae(left, right)}")

MAE = 2.667333687256246e-09


$(1.11)$ $$ u^i \cdot v_i = v_i \cdot u^i $$

In [167]:
left = to.contraction(u, to.lower_index(metric, v), [[0], [None]], [[None], [0]])
right = to.contraction(to.lower_index(metric, v), u, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.12)$ $$ u^i \cdot o_i = O $$

In [168]:
# (1.11) u · o = 0

left = to.contraction(u, to.lower_index(metric, o), [[0], [None]], [[None], [0]])
right = O

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.13)$  $$ u^i \cdot (\alpha \cdot v_i + \beta \cdot w_i) = \alpha \cdot (u^i \cdot v_i) + \beta \cdot (u^i \cdot w_i) $$ 

In [169]:
left = to.contraction(u, alpha*to.lower_index(metric, v) + beta*to.lower_index(metric, w), [[0], [None]], [[None], [0]])
right = alpha*to.contraction(u, to.lower_index(metric, v), [[0], [None]], [[None], [0]]) + beta*to.contraction(u, to.lower_index(metric, w), [[0], [None]], [[None], [0]])

print(f"MAE = {mae(left, right)}")

MAE = 1.874447974614668e-07


$(1.26)$ $$ u^i \times v^i = - (v^i \times u^i) $$

In [170]:
left = to.cross_prod(u, v, E)
right = -to.cross_prod(v, u, E)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.28)$ $$ (\alpha \cdot u^i) \times v^i = u^i \times (\alpha \cdot v^i) = \alpha \cdot (u^i \times v^i) $$

In [171]:
left = to.cross_prod(alpha*u, v, E)
center = to.cross_prod(u, alpha*v, E)
right = alpha*to.cross_prod(u, v, E)

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 2.404197729788393e-09
MAE = 3.1465201555391786e-09
MAE = 3.147747840159809e-09


$(1.29)$ $$ u_i \cdot (v^i \times w^i) = v_i \cdot (w^i \times u^i) = w_i \cdot (u^i \times v^i) $$

In [172]:
left = to.contraction(u, to.cross_prod(v, w, E), [[0], [None]], [[None], [0]])
center = to.contraction(v, to.cross_prod(w, u, E), [[0], [None]], [[None], [0]])
right = to.contraction(w, to.cross_prod(u, v, E), [[0], [None]], [[None], [0]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 3.9741843238516594e-07
MAE = 4.1612372569943545e-07
MAE = 4.214336399854801e-07


$(1.30)$ $$ u^i \times (v^i + w^i) = (u^i \times v^i) + (u^i \times w^i) $$

In [173]:
left = to.cross_prod(u, v + w, E)
right = to.cross_prod(u, v, E) + to.cross_prod(u, w, E)

print(f"MAE = {mae(left, right)}")

MAE = 1.7265431040414114e-07


$(1.31)$ $$ u_i \cdot (u^i \times v^i) = O $$

In [174]:
left = to.contraction(u, to.cross_prod(u, v, E), [[0], [None]], [[None], [0]])
right = O

print(f"MAE = {mae(left, right)}")

MAE = 6.489479176252644e-08


# 1.2 Algebra of tensors

$(1.49)$ $$ A^i_j \cdot (\alpha \cdot u^j + v^j)  = \alpha \cdot A^i_j \cdot u^j + A^i_j \cdot v^j $$

In [175]:
left = to.contraction(A, alpha*u + v, [[None], [0]], [[0], [None]])
right = alpha*to.contraction(A, u, [[None], [0]], [[0], [None]]) + to.contraction(A, v, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 1.3080607175197656e-07


$(1.50) \ [1]$ $$ (A^i_j + B^i_j) \cdot u^j =  A^i_j \cdot u^j + B^i_j \cdot u^j$$

In [176]:
# (1.50) (A + B)*u = A*u + B*u

left = to.contraction(A + B, u, [[None], [0]], [[0], [None]])
right = to.contraction(A, u, [[None], [0]], [[0], [None]]) + to.contraction(B, u, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 1.748278606328313e-07


$(1.50) \ [2]$ $$ (A^i_j - B^i_j) \cdot u^j =  A^i_j \cdot u^j - B^i_j \cdot u^j$$

In [177]:
# (1.50) (A - B)*u = A*u - B*u

left = to.contraction(A - B, u, [[None], [0]], [[0], [None]])
right = to.contraction(A, u, [[None], [0]], [[0], [None]]) - to.contraction(B, u, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 1.555261945895836e-07


$(1.51)$ $$ (\alpha \cdot A^i_j) \cdot u^j = \alpha \cdot (A^i_j \cdot u^j) $$

In [178]:
left = to.contraction(alpha*A, u, [[None], [0]], [[0], [None]])
right = alpha*to.contraction(A, u, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 5.894150945096044e-09


$(1.53)$ $$ (u^i \otimes v_j) \cdot w^j = u^i \cdot (v^j \cdot w_j) = (v^j \cdot w_j) \cdot u^i$$

In [179]:
# (1.53) (u xx v)*w = u*(v · w) = (v · w)*u

left = to.contraction(to.tensor_product(u, to.lower_index(metric, v)), w, [[None], [0]], [[0], [None]])
center = u*to.contraction(v, to.lower_index(metric, w), [[0], [None]], [[None], [0]])
right = to.contraction(v, to.lower_index(metric, w), [[0], [None]], [[None], [0]])*u

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 2.324674568399132e-07
MAE = 0.0
MAE = 2.324674568399132e-07


$(1.54)$ $$ (u^i \otimes v^j) \cdot (\alpha \cdot w_i + x_i) = \alpha \cdot(u^i \otimes v^j) \cdot w_i + (u^i \otimes v^j) \cdot x_i $$

In [180]:
left = to.contraction(to.tensor_product(u, v), (alpha*to.lower_index(metric, w) + to.lower_index(metric, x)), [[0], [None]], [[None], [0]])
right = alpha*to.contraction(to.tensor_product(u, v), to.lower_index(metric, w), [[0], [None]], [[None], [0]]) + \
        to.contraction(to.tensor_product(u, v), to.lower_index(metric, x), [[0], [None]], [[None], [0]])

print(f"MAE = {mae(left, right)}")

MAE = 1.4960043870360096e-07


$(1.55)$ $$ (\alpha \cdot u^i + \beta \cdot v^i) \otimes w^j = \alpha (u^i \otimes w^j) + \beta (v^i \otimes w^j)  $$

In [181]:
left = to.tensor_product((alpha*u + beta*v), w)
right = alpha*to.tensor_product(u, w) + beta*to.tensor_product(v, w)

print(f"MAE = {mae(left, right)}")

MAE = 6.987779244127523e-08


$(1.56)$ $$ u_i \cdot (v^i \otimes w^j) = (u_i \cdot v^i) \cdot w^j = w^j \cdot (u_i \cdot v^i) $$

In [182]:
left = to.contraction(to.lower_index(metric, u), to.tensor_product(v, w), [[None], [0]], [[0], [None]])
center = to.contraction(to.lower_index(metric, u), v, [[None], [0]], [[0], [None]])*w
right = w*to.contraction(to.lower_index(metric, u), v, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 2.1292862584232353e-07
MAE = 0.0
MAE = 2.1292862584232353e-07


$(1.57)$ $$ (u^i \otimes v_j) \cdot (w^j \otimes x_k) = (v_j \cdot w^j) \cdot u^i \otimes x_k = u^i \otimes x_k (v_j \cdot w^j) $$

In [183]:
left = to.contraction(to.tensor_product(u, to.lower_index(metric, v)), to.tensor_product(w, to.lower_index(metric, x)), [[None], [0]], [[0], [None]])
center = to.tensor_product(to.contraction(to.lower_index(metric, v), w, [[None], [0]], [[0], [None]])*u, to.lower_index(metric, x))
right = to.tensor_product(u, to.lower_index(metric, x)*to.contraction(to.lower_index(metric, v), w, [[None], [0]], [[0], [None]]))

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 1.655651317378215e-06
MAE = 8.101403068394575e-07
MAE = 1.6535776694581727e-06


$(1.58)$ $$ A^i_j \cdot (u^j \otimes v^k) = (A^i_j \cdot u^j) \otimes v^k $$  

In [184]:
left = to.contraction(A, to.tensor_product(u, v), [[None], [0]], [[0], [None]])
right = to.tensor_product(to.contraction(A, u, [[None], [0]], [[0], [None]]), v)

print(f"MAE = {mae(left, right)}")

MAE = 9.892352181850583e-08


$(1.71)$ $$ A^i_j + B^i_j = B^i_j + A^i_j$$

In [185]:
left = A + B
right = B + A

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.73)$ $$ A^i_j - A^i_j = OO $$

In [186]:
left = A + (-A)
right = OO

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.74)$ $$ A^i_j + B^i_j + C^i_j = (A^i_j + B^i_j) + C^i_j $$

In [187]:
left = A + (B + C)
right = (A + B) + C

print(f"MAE = {mae(left, right)}")

MAE = 9.481998830551674e-08


$(1.76)$ $$ (A^i_j \cdot B^k_i) \cdot C^l_k = A^i_j \cdot (B^k_i \cdot C^l_k) $$

In [188]:
left = to.contraction(to.contraction(A, B, [[0], [None]], [[None], [0]]), C, [[0], [None]], [[None], [0]])
right = to.contraction(A, to.contraction(B, C, [[0], [None]], [[None], [0]]), [[0], [None]], [[None], [0]])

print(f"MAE = {mae(left, right)}")

MAE = 1.6732978110667318e-05


$(1.78)$ $$ (A^i_j + B^i_j) \cdot C^j_k = A^i_j \cdot C^j_k + B^i_j \cdot C^j_k $$

In [189]:
left = to.contraction(A+B, C, [[None], [0]], [[0], [None]])
right = to.contraction(A, C, [[None], [0]], [[0], [None]]) + to.contraction(B, C, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 2.173478151235031e-06


$(1.81)$ $$ v^i \cdot (A^i_j)^T \cdot u_j = u_i \cdot A^i_j \cdot v^j = A^i_j \cdot v^j \cdot u^j $$

In [190]:
left = to.contraction(to.contraction(to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A)), covariant_index=0), u, [[None], [0]], [[0], [None]]), to.lower_index(metric, v), [[0], [None]], [[None], [0]])
center = to.contraction(to.lower_index(metric, u), to.contraction(A, v, [[None], [0]], [[0], [None]]), [[None], [0]], [[0], [None]])
right =  to.contraction(to.contraction(A, v, [[None], [0]], [[0], [None]]), to.lower_index(metric, u), [[0], [None]], [[None], [0]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 3.800159618094767e-07
MAE = 0.0
MAE = 3.800159618094767e-07


$(1.82)$ $$ [(A^i_j)^T]^T = A^i_j$$

In [191]:
A_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A)), covariant_index=0)
left = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A_T)), covariant_index=0)
right = A

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.83)$ $$ (\alpha \cdot A^i_j + \beta \cdot B^i_j)^T = \alpha \cdot (A^i_j)^T + \beta \cdot (B^i_j)^T $$

In [192]:
left = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, alpha*A + beta*B)), covariant_index=0)
A_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A)), covariant_index=0)
B_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, B)), covariant_index=0)
right = alpha*A_T + beta*B_T

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.84)$ $$ (A^i_j \cdot B^j_k)^T = (B^j_k)^T \cdot (A^i_j)^T $$

In [193]:
AB = to.contraction(A, B, [[None], [0]], [[0], [None]])
left = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, AB)), covariant_index=0)
A_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A)), covariant_index=0)
B_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, B)), covariant_index=0)
right = to.contraction(B_T, A_T, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.85)$ $$ (u^i \otimes v^j)^T = (v^j \otimes u^i) $$

In [194]:
left = to.transpose(to.tensor_product(u, v))
right = to.tensor_product(v, u)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.86)$ $$ \text{tr} (u^i \otimes v_j) = u^i \cdot v_i$$

In [195]:
left = to.trace(to.tensor_product(u, to.lower_index(metric, v)))
right = to.contraction(to.lower_index(metric, u), v, [[None], [0]], [[0], [None]])

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.89)$ $$ \text{tr}[(A^i_j)^T] = \text{tr}(A^i_j) $$ 

In [196]:
A_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A)), covariant_index=0)
left = to.trace(A_T)
right = to.trace(A)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.90)$ $$ \text{tr}(A^i_j \cdot B^j_k) = \text{tr}(B^i_j \cdot A^j_k) $$ 

In [197]:
left = to.trace(to.contraction(A, B, [[None], [0]], [[0], [None]]))
right = to.trace(to.contraction(B, A, [[None], [0]], [[0], [None]]))

print(f"MAE = {mae(left, right)}")

MAE = 3.2918336501097656e-07


$(1.91)$ $$ \text{tr}(A^i_j + B^i_j) = \text{tr}(A^i_j) + \text{tr}(B^i_j) $$

In [198]:
left = to.trace(A + B)
right = to.trace(A) + to.trace(B)

print(f"MAE = {mae(left, right)}")

MAE = 3.411742000025697e-07


$(1.92)$ $$ \text{tr}(\alpha \cdot A^i_j) = \alpha \cdot \text{tr}(A^i_j) $$

In [199]:
left = to.trace(alpha*A)
right = alpha*to.trace(A)

print(f"MAE = {mae(left, right)}")

MAE = 4.647622287023978e-09


$(1.93)$ $$ A^i_j : B^j_i = \text{tr} [(A^i_j)^T \cdot B^i_k] = B^i_j : A^j_i $$

In [200]:
A_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A)), covariant_index=0)
left = to.trace(to.contraction(A_T, B, [[None], [0]], [[0], [None]]))
center = to.trace(to.contraction(A_T, B, [[None], [0]], [[0], [None]]))
B_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, B)), covariant_index=0)
right = to.trace(to.contraction(B_T, A, [[None], [0]], [[0], [None]]))

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 0.0
MAE = 0.0
MAE = 0.0


$(1.94)$ $$ I^i_j : A^j_i = \text{tr} (A^i_j) = A^i_j : I^j_i $$

In [201]:
left = to.contraction(I, A, [[0], [0]], [[0], [0]])
center = to.trace(A)
right = to.contraction(A, I, [[0], [0]], [[0], [0]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 0.0
MAE = 0.0
MAE = 0.0


$(1.95)$ $$  A^i_k : (B^i_j \cdot C^j_k) = [(B^i_j)^T A^k_j] : C^i_k = [A^i_j \cdot (C^k_j)^T] \cdot B^k_i   $$

In [202]:
BC = to.contraction(B, C, [[None], [0]], [[0], [None]])
A_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A)), covariant_index=0)
left = to.trace(to.contraction(A_T, BC, [[None], [0]], [[0], [None]]))

B_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, B)), covariant_index=0)
BT_A = to.contraction(B_T, A, [[None], [0]], [[0], [None]])
BT_A_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, BT_A)), covariant_index=0)
center = to.trace(to.contraction(BT_A_T, C, [[None], [0]], [[0], [None]]))

C_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, C)), covariant_index=0)
A_CT = to.contraction(A, C_T, [[None], [0]], [[0], [None]])
A_CT_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A_CT)), covariant_index=0)
right = to.trace(to.contraction(A_CT_T, B, [[None], [0]], [[0], [None]]))

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 7.039136107778177e-06
MAE = 6.761551503586816e-06
MAE = 7.677516805415507e-06


$(1.96)$  $$ A_{ij} : (u^i \otimes v^j) = u^i \cdot A_{ij} \cdot v^j = (u^i \otimes v^j) : A_{ij}$$

In [203]:
left = to.contraction(to.lower_index(metric, A), to.tensor_product(u, v), [[None], [0, 1]], [[0, 1], [None]])
center = to.contraction(u, to.contraction(to.lower_index(metric, A), v, [[None], [1]], [[0], [None]]), [[0], [None]], [[None], [0]])
right = to.contraction(to.tensor_product(u, v), to.lower_index(metric, A), [[0, 1], [None]], [[None], [0, 1]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 4.160906996730773e-07
MAE = 4.160906996730773e-07
MAE = 0.0


$(1.97)$   $$ (u^i \otimes v^j) : (w_i \otimes x_j) = (u^i \cdot w_i) \cdot (v^j \cdot x_j) $$

In [204]:
left = to.contraction(to.tensor_product(u, v), to.tensor_product(to.lower_index(metric, w), to.lower_index(metric, x)), [[0, 1], [None]], [[None], [0, 1]])
right = to.contraction(u, to.lower_index(metric, w), [[0], [None]], [[None], [0]])*to.contraction(v, to.lower_index(metric, x), [[0], [None]], [[None], [0]])

print(f"MAE = {mae(left, right)}")

MAE = 3.298900537629379e-06


$(1.101)$   $$ \text{det}(A^i_j \cdot B^j_k) = \text{det}(A^i_j) \cdot \text{det}(B^j_k)$$

In [205]:
left = to.determinant(to.contraction(A, B, [[None], [0]], [[0], [None]]))
right = to.determinant(A)*to.determinant(B)

print(f"MAE = {mae(left, right)}")

MAE = 0.0006948432419449091


$(1.102)$   $$ \text{det}[(A^i_j)^T] = \text{det}(A^i_j)$$

In [206]:
# (1.102) det(At) = det(A)

A_T = to.raise_index(inv_metric, to.transpose(to.lower_index(metric, A)), covariant_index=0)
left = to.determinant(A_T)
right = to.determinant(A)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


In [207]:
Ab = to.lower_index(metric, A)
Bh = to.raise_index(inv_metric, B)

S = 1/2*(Ab + to.transpose(Ab))
W = 1/2*(Ab - to.transpose(Ab))

$(1.112)$  $$ S = \frac{1}{2}(A_{ij} + A_{ij}^T) $$

In [208]:
# (1.112) S = 1/2(A + At)

left = S
right = to.symmetrize(Ab, indices=[0, 1], order='covariant')

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.112)$  $$ W = \frac{1}{2}(A_{ij} - A_{ij}^T) $$

In [209]:
# (1.112) W = 1/2(A - At)

left = W
right = to.antisymmetrize(Ab, indices=[0, 1], order='covariant')

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.113)$  $$ S^i_j = (S^i_j)^T $$

In [210]:
left = S
right = to.transpose(S)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.113)$  $$ W^i_j = -(W^i_j)^T $$

In [211]:
left = W
right = -to.transpose(W)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.115)$  $$ S^i_j : B^j_i = S^i_j : (B^i_j)^T = S^i_j : \frac{1}{2}[B^j_i + (B^i_j)^T] $$

In [212]:
left = to.contraction(S, Bh, [[None], [0, 1]], [[0, 1], [None]])
center = to.contraction(S, to.transpose(Bh), [[None], [0, 1]], [[0, 1], [None]])
right = to.contraction(S, to.symmetrize(Bh, indices=[0, 1], order='contravariant'), [[None], [0, 1]], [[0, 1], [None]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 2.551633997427416e-07
MAE = 3.9646775462642836e-07
MAE = 4.025254725092964e-07


$(1.116)$  $$ W^i_j : B^j_i = -W^i_j : (B^i_j)^T = W^i_j : \frac{1}{2}[B^j_i - (B^i_j)^T] $$

In [213]:
left = to.contraction(W, Bh, [[None], [0, 1]], [[0, 1], [None]])
center = -to.contraction(W, to.transpose(Bh), [[None], [0, 1]], [[0, 1], [None]])
right = to.contraction(W, to.antisymmetrize(Bh, indices=[0, 1], order='contravariant'), [[None], [0, 1]], [[0, 1], [None]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 8.190162503751708e-08
MAE = 1.1266060084835772e-07
MAE = 1.222179122351008e-07


$(1.117)$ $$ S^i_j : W^j_i = O $$

In [214]:
left = to.contraction(S, to.raise_index(inv_metric, to.raise_index(inv_metric, W)), [[None], [0, 1]], [[0, 1], [None]])
right = O

print(f"MAE = {mae(left, right)}")

MAE = 5.029064595873933e-07


$(1.137)$  $$ (u^i \otimes v^j \otimes w^k) \cdot x_k = (w^k \cdot x_k) \cdot u^i \otimes v^j$$

In [215]:
left = to.contraction(to.tensor_product(to.tensor_product(u, v), w), to.lower_index(metric, x), [[2], [None]], [[None], [0]])
right = to.tensor_product(to.contraction(w, to.lower_index(metric, x), [[0], [None]], [[None], [0]])*u, v)

print(f"MAE = {mae(left, right)}")

MAE = 1.32454729850906e-07


$(1.139)$ $$ (u^i \otimes v^j \otimes w^k) \cdot (x_j \otimes y_k) = (v^j \cdot x_j) \cdot (w^k \cdot y_k) \cdot u^i$$

In [216]:
left = to.contraction(to.tensor_product(to.tensor_product(u, v), w), to.tensor_product(to.lower_index(metric, x), to.lower_index(metric, y)), [[1, 2], [None]], [[None], [0, 1]])
right = to.contraction(v, to.lower_index(metric, x), [[0], [None]], [[None], [0]])*to.contraction(w, to.lower_index(metric, y), [[0], [None]], [[None], [0]])*u

print(f"MAE = {mae(left, right)}")

MAE = 2.784997548133106e-07


$(1.140)$  $$ (u^i \otimes v_j \otimes w^k):I^j_k = (v_j \cdot w^j) \cdot u^i$$

In [217]:
left = to.contraction(to.tensor_product(to.tensor_product(u, to.lower_index(metric, v)), w), I, [[1], [0]], [[0], [0]])
right = to.contraction(to.lower_index(metric, v), w, [[None], [0]], [[0], [None]])*u

print(f"MAE = {mae(left, right)}")

MAE = 2.324674568399132e-07


$(1.146)$  $$ E_{ijk}:(u^i \otimes v^j) = v^i \times u^j $$

In [218]:
left = to.contraction(E, to.tensor_product(u, v), [[None], [0, 1]], [[1, 0], [None]])
right = to.cross_prod(v, u, E)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


$(1.152)$ $$ (A^i_j \otimes B^k_l):C^l_k = A^i_j \cdot (B^k_l : C^l_k) = (B^k_l : C^l_k) \cdot A^i_j $$

In [219]:
left = to.contraction(to.tensor_product(A, B), C, [[1], [1]], [[0], [0]])
center = A*to.contraction(B, C, [[0], [0]], [[0], [0]])
right = to.contraction(B, C, [[0], [0]], [[0], [0]])*A

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 3.873320383718237e-05
MAE = 0.0
MAE = 3.873320383718237e-05


$(1.153)$  $$ A^j_i : (B^i_j \otimes C^k_l) = (A^k_l : B^l_k) \cdot C^i_j = C^i_j \cdot (A^k_l : B^l_k)$$

In [220]:
left = to.contraction(A, to.tensor_product(B, C), [[0], [0]], [[0], [0]])
center = to.contraction(A, B, [[0], [0]], [[0], [0]])*C
right = C*to.contraction(A, B, [[0], [0]], [[0], [0]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 2.511604861865635e-06
MAE = 0.0
MAE = 2.511604861865635e-06


$(1.154)$  $$ (A^i_j \otimes B^k_l) : (C^l_k \otimes D^n_m) = (B^k_l : C^l_k) \cdot (A^i_j \otimes D^n_m) = (A^i_j \otimes D^n_m) \cdot (B^k_l : C^l_k)$$

In [221]:
left = to.contraction(to.tensor_product(A, B), to.tensor_product(C, D), [[1], [1]], [[0], [0]])
center = to.contraction(B, C, [[0], [0]], [[0], [0]])*to.tensor_product(A, D)
right = to.tensor_product(A, D)*to.contraction(B, C, [[0], [0]], [[0], [0]])

print(f"MAE = {mae(left, center)}")
print(f"MAE = {mae(center, right)}")
print(f"MAE = {mae(left, right)}")

MAE = 0.00015598158643115312
MAE = 0.0
MAE = 0.00015598158643115312


$(1.155)$  $$ A^i_j \cdot (B^j_l \otimes C^n_m) \cdot D^m_q = (A^i_j \cdot B^j_l) \otimes (C^n_m \cdot D^m_q) $$

In [222]:
left = to.contraction(to.contraction(A, to.tensor_product(B, C), [[0], [None]], [[None], [0]]), D, [[1], [None]], [[None], [0]])
right = to.tensor_product(to.contraction(A, B, [[0], [None]], [[None], [0]]), to.contraction(C, D, [[0], [None]], [[None], [0]]))

print(f"MAE = {mae(left, right)}")

MAE = 0.00015424037701450288
